In [ ]:
import os
os.environ["GIT_PYTHON_REFRESH"] = "quiet"
import git
from myosuite.utils import gym
import skvideo
skvideo.setFFmpegPath(r"C:\Users\rohan\Downloads\ffmpeg-2025-09-04-git-2611874a50-essentials_build\ffmpeg-2025-09-04-git-2611874a50-essentials_build\bin")
import skvideo.io
import numpy as np
import myosuite
from stable_baselines3 import PPO
import matplotlib.pyplot as plt
import time
from IPython.display import HTML
from base64 import b64encode
import PIL.Image, PIL.ImageDraw

def show_video(video_path, video_width=400):
    video_file = open(video_path, "r+b").read()
    video_url = f'data:video/mp4;base64,{b64encode(video_file).decode()}'
    return HTML(f"<video autoplay width={video_width} controls><source src=\"{video_url}\"></video>")

def add_text_to_frame(frame, text, pos=(20, 20), color=(255, 0, 0), fontsize=12):
    if isinstance(frame, np.ndarray):
        frame = PIL.Image.fromarray(frame)
    draw = PIL.ImageDraw.Draw(frame)
    draw.text(pos, text, fill=color)
    return frame

from stable_baselines3 import PPO
import gymnasium as gym

In [3]:
import sys
sys.path.append(r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist")

In [ ]:
import sys
sys.argv = [sys.argv[0]]


In [ ]:
!python myoassist\ctrl_optim\run_optim.py baseline_cie



In [ ]:
import os
import sys
import subprocess
import glob
import platform
import shutil
from pathlib import Path
from IPython.display import display, clear_output
import time
import threading
from queue import Queue, Empty

os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['PYTHON_EXECUTABLE'] = sys.executable


class JupyterTrainingRunner:
    def __init__(self):
        self.script_dir = self.get_script_directory()
        self.original_dir = os.getcwd()
        
    def get_script_directory(self):
        return Path(r"c:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\ctrl_optim").absolute()
        if '__file__' in globals():
            return Path(__file__).parent.absolute()
        else:
            return Path.cwd()
    
    def list_available_configs(self):
        config_dir = self.script_dir / "optim" / "training_configs"
        
        if not config_dir.exists():
            return []
        
        config_files = []
        for ext in ["*.bat", "*.sh"]:
            config_files.extend(glob.glob(str(config_dir / ext)))
        
        if not config_files:
            return []
        
        config_names = []
        for config_file in config_files:
            name = Path(config_file).stem
            if name not in config_names:
                config_names.append(name)
        
        return sorted(config_names)
    
    def setup_environment(self):
        os.chdir(self.script_dir.parent)
        
        root_dir = Path(os.getcwd()) / "ctrl_optim"
        
        current_pythonpath = os.environ.get('PYTHONPATH', '')
        new_pythonpath = f"{root_dir}{os.pathsep}{current_pythonpath}" if current_pythonpath else str(root_dir)
        
        os.environ['PYTHONPATH'] = new_pythonpath
        os.environ['ROOT_DIR'] = str(root_dir)
        os.environ['PYTHONUNBUFFERED'] = '1'
        
        os.chdir(root_dir)
    
    def read_output_stream(self, stream, queue):
        try:
            for line in iter(stream.readline, ''):
                if line:
                    queue.put(line.rstrip())
            queue.put(None)
        except Exception as e:
            queue.put(f"Error reading stream: {e}")
            queue.put(None)
    
    def run_training_with_realtime_output(self, config_name, update_interval=0.1):
        config_dir = self.script_dir / "optim" / "training_configs"
        
        is_windows = platform.system() == "Windows"
        
        if is_windows:
            primary_config = config_dir / f"{config_name}.bat"
            fallback_config = config_dir / f"{config_name}.sh"
        else:
            primary_config = config_dir / f"{config_name}.sh"
            fallback_config = config_dir / f"{config_name}.bat"
        
        config_file = None
        if primary_config.exists():
            config_file = primary_config
        elif fallback_config.exists():
            config_file = fallback_config
        
        if not config_file:
            return False
        
        optim_dir = self.script_dir / "optim"
        os.chdir(optim_dir)
        
        try:
            if platform.system() == "Windows":
                if config_file.suffix == ".bat":
                    cmd = ["cmd", "/c", str(config_file)]
                else:
                    if not shutil.which('bash'):
                        return False
                    cmd = ["bash", str(config_file)]
            else:
                if config_file.suffix == ".sh":
                    os.chmod(config_file, 0o755)
                    cmd = ["bash", str(config_file)]
                else:
                    return False
            
            process = subprocess.Popen(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                universal_newlines=True,
                bufsize=1,
                env=os.environ.copy()
            )
            
            output_queue = Queue()
            
            output_thread = threading.Thread(
                target=self.read_output_stream,
                args=(process.stdout, output_queue)
            )
            output_thread.daemon = True
            output_thread.start()
            
            output_lines = []
            last_update = time.time()
            
            while True:
                try:
                    line = output_queue.get(timeout=update_interval)
                    
                    if line is None:
                        break
                    
                    output_lines.append(line)
                    
                    current_time = time.time()
                    if (current_time - last_update > update_interval or 
                        any(keyword in line.lower() for keyword in 
                            ['error', 'iteration', 'epoch', 'generation', 'best', 'completed'])):
                        
                        clear_output(wait=True)
                        
                        display_lines = output_lines[-50:] if len(output_lines) > 50 else output_lines
                        for display_line in display_lines:
                            print(display_line)
                        
                        last_update = current_time
                        
                except Empty:
                    if process.poll() is not None:
                        break
                    continue
            
            process.wait()
            
            clear_output(wait=True)
            
            display_lines = output_lines[-100:] if len(output_lines) > 100 else output_lines
            for line in display_lines:
                print(line)
            
            if process.returncode == 0:
                return True
            else:
                return False
                
        except Exception as e:
            return False
        finally:
            os.chdir(self.original_dir)
    
    def run_training_simple(self, config_name):
        config_dir = self.script_dir / "optim" / "training_configs"
        
        is_windows = platform.system() == "Windows"
        
        if is_windows:
            config_file = config_dir / f"{config_name}.bat"
            if not config_file.exists():
                config_file = config_dir / f"{config_name}.sh"
        else:
            config_file = config_dir / f"{config_name}.sh"
            if not config_file.exists():
                config_file = config_dir / f"{config_name}.bat"
        
        if not config_file.exists():
            return False
        
        optim_dir = self.script_dir / "optim"
        os.chdir(optim_dir)
        
        try:
            if platform.system() == "Windows":
                if config_file.suffix == ".bat":
                    result = subprocess.run(["cmd", "/c", str(config_file)], 
                                          shell=True, check=True, 
                                          capture_output=True, text=True,
                                          env=os.environ.copy())
                else:
                    result = subprocess.run(["bash", str(config_file)], 
                                          shell=True, check=True, 
                                          capture_output=True, text=True,
                                          env=os.environ.copy())
            else:
                os.chmod(config_file, 0o755)
                result = subprocess.run([str(config_file)], 
                                      shell=True, check=True, 
                                      capture_output=True, text=True,
                                      env=os.environ.copy())
            
            print(result.stdout)
            if result.stderr:
                print(result.stderr)
            
            return True
            
        except subprocess.CalledProcessError as e:
            if e.stdout:
                print(e.stdout)
            if e.stderr:
                print(e.stderr)
            return False
        finally:
            os.chdir(self.original_dir)


def run_training(config_name, realtime=True, update_interval=0.1):
    runner = JupyterTrainingRunner()
    runner.setup_environment()
    
    if realtime:
        return runner.run_training_with_realtime_output(config_name, update_interval)
    else:
        return runner.run_training_simple(config_name)


def list_configs():
    runner = JupyterTrainingRunner()
    configs = runner.list_available_configs()
    
    if configs:
        for i, config in enumerate(configs, 1):
            print(f"{i:2d}. {config}")
    
    return configs

In [2]:
import os
run_training('baseline_hmedi', realtime = True)

Training: baseline_hmedi
R Exo CTRL: -0.02, L Exo CTRL: -0.04
R Exo CTRL: -0.02, L Exo CTRL: -0.04
R Exo CTRL: -0.02, L Exo CTRL: -0.04
R Exo CTRL: -0.02, L Exo CTRL: -0.04
R Exo CTRL: -0.02, L Exo CTRL: -0.00
R Exo CTRL: -0.02, L Exo CTRL: -0.01
R Exo CTRL: -0.02, L Exo CTRL: -0.01
R Exo CTRL: -0.02, L Exo CTRL: -0.01
R Exo CTRL: -0.02, L Exo CTRL: -0.01
R Exo CTRL: -0.02, L Exo CTRL: -0.01
R Exo CTRL: -0.02, L Exo CTRL: -0.01
R Exo CTRL: -0.02, L Exo CTRL: -0.01
R Exo CTRL: -0.02, L Exo CTRL: -0.00
R Exo CTRL: -0.02, L Exo CTRL: -0.01
R Exo CTRL: -0.02, L Exo CTRL: -0.01
R Exo CTRL: -0.03, L Exo CTRL: -0.01
R Exo CTRL: -0.03, L Exo CTRL: -0.01
R Exo CTRL: -0.03, L Exo CTRL: -0.01
R Exo CTRL: -0.03, L Exo CTRL: -0.01
R Exo CTRL: -0.03, L Exo CTRL: -0.01
R Exo CTRL: -0.03, L Exo CTRL: -0.01
R Exo CTRL: -0.04, L Exo CTRL: -0.00
R Exo CTRL: -0.04, L Exo CTRL: -0.01
R Exo CTRL: -0.04, L Exo CTRL: -0.01
R Exo CTRL: -0.04, L Exo CTRL: -0.01
R Exo CTRL: -0.05, L Exo CTRL: -0.01
Exception ign

KeyboardInterrupt: 

In [68]:
!python myoassist\ctrl_optim\run_eval.py


MyoSuite:> Registering Myo Envs


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\gymnasium\envs\registration.py:694: UserWarning: WARN: Overriding environment myoArmReachFixed-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\gymnasium\envs\registration.py:694: UserWarning: WARN: Overriding environment myoSarcArmReachFixed-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry."

In [ ]:
import os
from ctrl_optim.optim.optim_utils.fati_config_parser import load_params_and_create_testenv, print_config_summary
import time

PARAMS_FILE_PATH = r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\ctrl_optim\results\optim_results\hmedi_1000_0910_2316\myorfl_Kine_2D_1_25_2025Sep10_2316_None_BestLast.txt"

SIMULATION_TIME = 5
SLOPE_DEG = 0
MODEL = "hmedi"
EXO_BOOL = True
USE_4PARAM_SPLINE = True
N_POINTS = 4
MAX_TORQUE = 100

results_dir = os.path.dirname(PARAMS_FILE_PATH)
filename = os.path.basename(PARAMS_FILE_PATH)
bat_file_path = r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\ctrl_optim\optim\training_configs\baseline_hmedi.bat"

env, config, _ = load_params_and_create_testenv(
    results_dir=results_dir,
    filename=filename,
    bat_file_path=bat_file_path,
    sim_time=SIMULATION_TIME
)

env.reset()
env.env.mj_render()
time.sleep(3)
for i in range(200):
    obs, reward, done, truncated, info = env.step()
    env.env.mj_render()
    if done:
        break

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


MyoSuite:> Registering Myo Envs


c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\gymnasium\envs\registration.py:694: UserWarning: WARN: Overriding environment myoArmReachFixed-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\gymnasium\envs\registration.py:694: UserWarning: WARN: Overriding environment myoSarcArmReachFixed-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\gymnasium\envs\registration.py:694: UserWarning: WARN: Overriding environment myoFatiArmReachFixed-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\gymnasium\envs\registration.py:694: UserWarning: WARN: Overriding environment myoArmReachRandom-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registr

Using fatigue environment


ValueError: Error creating TestEnv with configuration: Number of spline points must be at least 1

In [ ]:
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
import mujoco

class ReflexExoWrapper(gym.Env):
    
    def __init__(self, reflex_env, fatigue_indices=None, fatigue_range=(0.7, 1.0), debug=False):
        super(ReflexExoWrapper, self).__init__()
        
        self.reflex_env = reflex_env
        self.base_env = reflex_env.env
        self.debug = debug
        
        self.n_muscles = 22
        self.fatigue_indices = fatigue_indices
        self.fatigue_range = fatigue_range
        
        self.apply_fatigue = not (fatigue_range == (0.0, 0.0) or fatigue_range == (1.0, 1.0))
        
        if self.apply_fatigue:
            self._init_cumulative_fatigue_dynamics()
        else:
            self._MA = np.zeros(self.n_muscles)
            self._MR = np.ones(self.n_muscles)
            self._MF = np.zeros(self.n_muscles)
            self.TL = np.zeros(self.n_muscles)
        
        self.action_space = gym.spaces.Box(
            low=0, 
            high=1.0, 
            shape=(2,), 
            dtype=np.float32
        )
        
        self.observation_space = self.base_env.observation_space

    def _init_cumulative_fatigue_dynamics(self):
        self._r = 10 * 15
        self._F = 0.00912
        self._R = 0.1 * 0.00094
        
        mj_model = None
        frame_skip = 1
        
        try:
            mj_model = self.base_env.unwrapped.model
        except AttributeError:
            try:
                mj_model = self.base_env.model
            except AttributeError:
                try:
                    mj_model = self.base_env.sim.model
                except AttributeError:
                    try:
                        mj_model = self.reflex_env.model
                    except AttributeError:
                        try:
                            mj_model = self.reflex_env.env.model
                        except AttributeError:
                            try:
                                mj_model = self.reflex_env.env.sim.model
                            except AttributeError:
                                pass
        
        if mj_model is not None:
            try:
                frame_skip = getattr(self.base_env.unwrapped, 'frame_skip', 1)
            except:
                frame_skip = getattr(self.base_env, 'frame_skip', 1)
            
            self._dt = mj_model.opt.timestep * frame_skip
            
            try:
                muscle_act_ind = mj_model.actuator_dyntype == mujoco.mjtDyn.mjDYN_MUSCLE
                self._tauact = np.array([mj_model.actuator_dynprm[i][0] for i in range(len(muscle_act_ind)) if muscle_act_ind[i]])
                self._taudeact = np.array([mj_model.actuator_dynprm[i][1] for i in range(len(muscle_act_ind)) if muscle_act_ind[i]])
            except:
                self._tauact = np.full(self.n_muscles, 0.01)
                self._taudeact = np.full(self.n_muscles, 0.04)
        else:
            self._dt = 0.01
            self._tauact = np.full(self.n_muscles, 0.01)
            self._taudeact = np.full(self.n_muscles, 0.04)
        
        if len(self._tauact) < self.n_muscles:
            self._tauact = np.full(self.n_muscles, 0.01)
            self._taudeact = np.full(self.n_muscles, 0.04)
        else:
            self._tauact = self._tauact[:self.n_muscles]
            self._taudeact = self._taudeact[:self.n_muscles]
        
        self._MA = np.zeros(self.n_muscles)
        self._MR = np.ones(self.n_muscles)
        self._MF = np.zeros(self.n_muscles)
        self.TL = np.zeros(self.n_muscles)

    def _compute_fatigue_MF(self, muscle_actions):
        self.TL = muscle_actions.copy()
        self._LD = 1/self._tauact * (0.5 + 1.5*self._MA)
        self._LR = 1/self._taudeact
        
        C = np.zeros_like(self._MA)
        idxs = (self._MA < self.TL) & (self._MR > (self.TL - self._MA))
        C[idxs] = self._LD[idxs] * (self.TL[idxs] - self._MA[idxs])
        idxs = (self._MA < self.TL) & (self._MR <= (self.TL - self._MA))
        C[idxs] = self._LD[idxs] * self._MR[idxs]
        idxs = self._MA >= self.TL
        C[idxs] = self._LR[idxs] * (self.TL[idxs] - self._MA[idxs])

        rR = np.zeros_like(self._MA)
        idxs = self._MA >= self.TL
        rR[idxs] = self._r*self._R
        idxs = self._MA < self.TL
        rR[idxs] = self._R

        C = np.clip(C, np.maximum(-self._MA/self._dt + self._F*self._MA, (self._MR - 1)/self._dt + rR*self._MF),
                    np.minimum((1 - self._MA)/self._dt + self._F*self._MA, self._MR/self._dt + rR*self._MF))

        dMA = (C - self._F*self._MA)*self._dt
        dMR = (-C + rR*self._MF)*self._dt
        dMF = (self._F*self._MA - rR*self._MF)*self._dt
        self._MA += dMA
        self._MR += dMR
        self._MF += dMF

        return self._MF.copy()

    def _apply_fatigue_scaling(self, muscle_actions):
        if not self.apply_fatigue:
            return muscle_actions.copy()
        
        MF_values = self._compute_fatigue_MF(muscle_actions)
        fatigued_muscle_actions = muscle_actions.copy()
        
        if self.fatigue_indices is not None:
            for idx in self.fatigue_indices:
                if idx < len(fatigued_muscle_actions):
                    strength_factor = 1.0 - MF_values[idx]
                    strength_factor = np.clip(strength_factor, self.fatigue_range[0], self.fatigue_range[1])
                    fatigued_muscle_actions[idx] *= strength_factor
        else:
            strength_factors = 1.0 - MF_values
            strength_factors = np.clip(strength_factors, self.fatigue_range[0], self.fatigue_range[1])
            fatigued_muscle_actions *= strength_factors
        
        return fatigued_muscle_actions

    def _get_reflex_actions(self):
        self.reflex_env.get_sensor_data()
        reflex_output = self.reflex_env.update(self.reflex_env.SENSOR_DATA)
        muscle_actions = self.reflex_env.reflex2mujoco(reflex_output)[:self.n_muscles]
        return muscle_actions

    def reset(self, **kwargs):
        self.reflex_env.reset()
        
        if self.apply_fatigue:
            self._MA = np.zeros(self.n_muscles)
            self._MR = np.ones(self.n_muscles)
            self._MF = np.zeros(self.n_muscles)
            self.TL = np.zeros(self.n_muscles)
        
        try:
            base_obs = self.base_env._get_obs()
        except AttributeError:
            base_obs = self.base_env.unwrapped.get_obs()
        
        return base_obs, {}

    def step(self, exo_actions, all_f=False):
        exo_actions = np.array(exo_actions, dtype=np.float32).reshape(-1)
        
        if len(exo_actions) != 2:
            raise ValueError(f"Expected 2 exoskeleton actions, got {len(exo_actions)}")
        
        is_done = False
        
        if not self.apply_fatigue:
            self.reflex_env.get_sensor_data()
            reflex_output = self.reflex_env.update(self.reflex_env.SENSOR_DATA)
            muscle_actions = self.reflex_env.reflex2mujoco(reflex_output)[:self.n_muscles]
            full_actions = np.concatenate([muscle_actions, exo_actions])
        else:
            muscle_actions = self._get_reflex_actions()
            processed_muscle_actions = self._apply_fatigue_scaling(muscle_actions)
            full_actions = np.concatenate([processed_muscle_actions, exo_actions])
        
        self.act = full_actions
        obs, reward, _, truncated, info = self.base_env.step(full_actions)
        
        self.reflex_env.update_footstep()
        
        body_xquat = self.base_env.sim.data.body('pelvis').xquat.copy()
        world_com_xpos = self.base_env.sim.data.body('pelvis').xpos.copy()
        pelvis_euler = self.reflex_env.get_intrinsic_EulerXYZ(body_xquat)
        
        if world_com_xpos[2] < 0.65:
            is_done = True
        
        if pelvis_euler[1] < np.deg2rad(-60) or pelvis_euler[1] > np.deg2rad(60):
            is_done = True
        
        return obs, (reward * -1), is_done, truncated, info

    def render(self, mode='human'):
        return self.reflex_env.render(mode=mode)
    
    def mj_render(self):
        return self.base_env.mj_render()

    def close(self):
        self.reflex_env.close()
    
    @property
    def muscle_active(self):
        return self._MA.copy()
    
    @property
    def muscle_resting(self):
        return self._MR.copy()
    
    @property
    def muscle_fatigue(self):
        return self._MF.copy()
    
    @property
    def target_load(self):
        return self.TL.copy()

Using fatigue environment

Loaded Configuration:
Using fatigue environment
  sim_time            : 60
  mode                : 2D
  init_pose           : walk_left
  slope_deg           : 0.0
  delayed             : False
  exo_bool            : False
  fixed_exo           : False
  n_points            : 0
  use_4param_spline   : False
  max_torque          : 0.0
  model               : dephy
  model_path          : None

Muscle forces: [ -83.6545  -12.5255 -288.7192  -11.7639  -76.966 ]
Muscle forces: [ -77.4118  -74.6673 -465.3891  -12.2774  -71.7995]
Muscle forces: [ -69.8543 -124.9792 -435.2911  -12.9265  -69.6205]
Muscle forces: [ -63.2129 -143.2146 -419.2592  -13.6464  -68.7162]
Muscle forces: [ -58.9646 -161.7097 -450.6478  -14.195   -67.0813]
Muscle forces: [ -55.7601 -178.5303 -616.3952  -13.9451  -65.4644]
Muscle forces: [ -53.2347 -195.9925 -693.2965  -13.8514  -67.9794]
Muscle forces: [ -51.8954 -208.1056 -721.9993  -13.8373  -78.946 ]
Muscle forces: [ -50.5989 -234.098  -72

In [ ]:
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO

class ReflexExoWrapper(gym.Env):
    
    def __init__(self, reflex_env, fatigue_indices=None, fatigue_range=(0.7, 1.0), debug=False):
        super(ReflexExoWrapper, self).__init__()
        
        self.reflex_env = reflex_env
        self.base_env = reflex_env.env
        self.debug = debug
        
        self.n_muscles = 22
        self.fatigue_indices = fatigue_indices
        self.fatigue_range = fatigue_range
        
        self.action_space = gym.spaces.Box(
            low=-1.0, 
            high=1.0, 
            shape=(2,), 
            dtype=np.float32
        )
        
        self.observation_space = self.base_env.observation_space

    def _apply_fatigue(self):
        MF = np.zeros(self.n_muscles)
        
        MF[self.fatigue_indices] = np.random.uniform(
            self.fatigue_range[0], 
            self.fatigue_range[1], 
            size=len(self.fatigue_indices)
        )
        
        self.base_env.unwrapped.muscle_fatigue.MF[:] = MF
        return MF

    def _apply_all_fatigue(self):
        MF = np.zeros(self.n_muscles)
        MF[:] = np.random.uniform(
            self.fatigue_range[0], 
            self.fatigue_range[1], 
            size=self.n_muscles
        )
        
        self.base_env.unwrapped.muscle_fatigue.MF[:] = MF
        return MF

    def _get_reflex_actions(self):
        self.reflex_env.get_sensor_data()
        reflex_output = self.reflex_env.update(self.reflex_env.SENSOR_DATA)
        muscle_actions = self.reflex_env.reflex2mujoco(reflex_output)[:self.n_muscles]
        return muscle_actions

    def reset(self, **kwargs):
        self.reflex_env.reset()
        self._apply_fatigue()
        
        try:
            base_obs = self.base_env._get_obs()
        except AttributeError:
            base_obs = self.base_env.unwrapped.get_obs()
        
        return base_obs, {}

    def step(self, exo_actions, all_f=False):
        if all_f:
            self._apply_all_fatigue()
        else:
            self._apply_fatigue()

        exo_actions = np.array(exo_actions, dtype=np.float32).reshape(-1)
        
        if len(exo_actions) != 2:
            raise ValueError(f"Expected 2 exoskeleton actions, got {len(exo_actions)}")
        
        is_done = False
        
        muscle_actions = self._get_reflex_actions()
        full_actions = np.concatenate([muscle_actions, exo_actions])
        
        obs, reward, _, truncated, info = self.base_env.step(full_actions)
        self.reflex_env.update_footstep()
        
        body_xquat = self.base_env.sim.data.body('pelvis').xquat.copy()
        world_com_xpos = self.base_env.sim.data.body('pelvis').xpos.copy()
        pelvis_euler = self.reflex_env.get_intrinsic_EulerXYZ(body_xquat)
        
        if world_com_xpos[2] < 0.65:
            is_done = True
        
        if pelvis_euler[1] < np.deg2rad(-60) or pelvis_euler[1] > np.deg2rad(60):
            is_done = True
        
        return obs, reward, is_done, truncated, info

    def render(self, mode='human'):
        return self.reflex_env.render(mode=mode)
    
    def mj_render(self):
        return self.base_env.mj_render()

    def close(self):
        self.reflex_env.close()

ReflexExoWrapper class defined successfully!

Key features:
- Frozen reflex controller handles first 22 muscle actions
- RL agent controls last 2 exoskeleton actions
- Fatigue applied to specified muscles each step
- Compatible with your reflex environment structure
- Includes termination conditions from your evaluation loop


In [ ]:
import os
import numpy as np
from ctrl_optim.optim.optim_utils.fati_config_parser import load_params_and_create_testenv, print_config_summary

PARAMS_FILE_PATH = r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\ctrl_optim\results\optim_results\dephy_0906_1958\myorfl_Kine_2D_1_25_2025Sep06_1958_None_BestLast.txt"
PARAMS_FILE_PATH = r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\ctrl_optim\results\optim_results\dephy_0907_1648\myorfl_Kine_2D_1_25_2025Sep07_1648_None_BestLast.txt"

SIMULATION_TIME = 5
SLOPE_DEG = 0
MODEL = "dephy"
EXO_BOOL = True
USE_4PARAM_SPLINE = False
N_POINTS = 4
MAX_TORQUE = 100

results_dir = os.path.dirname(PARAMS_FILE_PATH)
filename = os.path.basename(PARAMS_FILE_PATH)
bat_file_path = r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\ctrl_optim\optim\training_configs\baseline_hmedi.bat"

env, config, _ = load_params_and_create_testenv(
    results_dir=results_dir,
    filename=filename,
    bat_file_path=bat_file_path,
    sim_time=SIMULATION_TIME
)

fatigue_indices =  [5,6,16,17]
fatigue_range = (.1,.1)

env = ReflexExoWrapper(
    reflex_env=env,
    fatigue_indices=fatigue_indices,
    fatigue_range=fatigue_range,
    debug=True
)

from stable_baselines3 import PPO

model = PPO.load(r"ReflexExo_PPO_policy_dephy", env=env)

env.mj_render()
obs, info = env.reset()

import time
time.sleep(3)
tr = 0
for i in range(300):
    env.debug = False
    env.mj_render()
    
    exo_actions = model.predict(obs, deterministic=True)
    exo_actions = exo_actions[0].flatten()
    exo_actions = [1,1]
    
    obs, reward, done, truncated, info = env.step(exo_actions)
    tr += reward
    
    if done:
        break

Fixed ReflexExoWrapper with Conditional Fatigue Application

Key fixes:
- CumulativeFatigue dynamics ONLY run when fatigue_range != (0.0, 0.0) and != (1.0, 1.0)
- No interference with base environment when fatigue is disabled
- Frozen reflex controller handles first 22 muscle actions
- RL agent controls last 2 exoskeleton actions
- Clean separation between fatigue and no-fatigue modes

Usage:
- No fatigue: fatigue_range=(0.0, 0.0) or (1.0, 1.0)
- Light fatigue: fatigue_range=(0.7, 1.0)
- Moderate fatigue: fatigue_range=(0.4, 0.8)
- Severe fatigue: fatigue_range=(0.1, 0.5)
- All muscles: fatigue_indices=None
- Specific muscles: fatigue_indices=[0, 4, 11, 15]


In [ ]:
import os
import numpy as np
from stable_baselines3 import PPO
from ctrl_optim.optim.optim_utils.fati_config_parser import load_params_and_create_testenv, print_config_summary

PARAMS_FILE_PATH = r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\ctrl_optim\results\optim_results\dephy_0907_1648\myorfl_Kine_2D_1_25_2025Sep07_1648_None_BestLast.txt"

SIMULATION_TIME = 10

Exception ignored in: <function Renderer.__del__ at 0x000002B295F73F60>
Traceback (most recent call last):
  File "c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\myosuite\renderer\renderer.py", line 142, in __del__
    self.close()
  File "c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\myosuite\renderer\mj_renderer.py", line 158, in close
    quit()
    ^^^^
NameError: name 'quit' is not defined


Using fatigue environment

Loaded Configuration:
Using fatigue environment
  sim_time            : 5
  mode                : 2D
  init_pose           : walk_left
  slope_deg           : 0.0
  delayed             : False
  exo_bool            : False
  fixed_exo           : False
  n_points            : 0
  use_4param_spline   : False
  max_torque          : 0.0
  model               : hmedi
  model_path          : None


=== FATIGUE CONFIGURATION ===
Fatigue indices: [5, 6, 16, 17]
Fatigue range: (0.1, 0.1) (Fatigue enabled)
Fatigue will be applied to 4 specific muscles: [5, 6, 16, 17]
CumulativeFatigue dynamics initialized:
  - dt: 0.01
  - F: 0.00912, R: 9.400000000000001e-05, r: 150
  - tauact shape: (22,)
  - taudeact shape: (22,)
Reflex Exo Wrapper initialized:
  - Number of muscles: 22
  - Fatigue indices: [5, 6, 16, 17]
  - Fatigue range: (0.1, 0.1)
  - Apply fatigue: True
  - Action space (exo only): (2,)
  - Observation space: (134,)
Wrapping the env with a `Monitor` wrapper
W

In [ ]:
import gymnasium as gym
import numpy as np
import time

gym.register(
    id="myoLegCustomWalk-v0",
    entry_point="myosuite.envs.myo.myobase.walk_v0:WalkEnvV0",
    max_episode_steps=1000,
    kwargs={
        "model_path": r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\models\80muscle\myoLeg80_HMEDI\myolegs_HMEDI.xml",
        "normalize_act": True,
        "min_height": 0.8,
        "max_rot": 0.8,
        "hip_period": 100,
        "reset_type": "init",
        "target_x_vel": 0.0,
        "target_y_vel": 1.2,
        "target_rot": None,
    },
)

def test_walk_environment():
    try:
        env = gym.make("myoLegCustomWalk-v0")
        obs, info = env.reset()
        
        total_reward = 0
        episode_length = 100
        
        for step in range(episode_length):
            action = env.action_space.sample()
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            total_reward += reward
            
            if step % 20 == 0:
                env.render()
            
            time.sleep(0.01)
            
            if done:
                break
        
        obs, info = env.reset()
        env.close()
        return True
        
    except Exception as e:
        return False

def quick_walk_test():
    try:
        env = gym.make("myoLegCustomWalk-v0")
        obs, info = env.reset()
        
        for i in range(10):
            action = env.action_space.sample()
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            if done:
                break
        
        env.close()
        return True
        
    except Exception as e:
        return False

if __name__ == "__main__":
    basic_success = quick_walk_test()
    
    if basic_success:
        test_walk_environment()

Using fatigue environment

Loaded Configuration:
Using fatigue environment
  sim_time            : 10
  mode                : 2D
  init_pose           : walk_left
  slope_deg           : 0.0
  delayed             : False
  exo_bool            : False
  fixed_exo           : False
  n_points            : 0
  use_4param_spline   : False
  max_torque          : 0.0
  model               : hmedi
  model_path          : None

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to ./ppo_reflex_exo_tensorboard/PPO_9
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 88.9     |
|    ep_rew_mean     | 3.3e+03  |
| time/              |          |
|    fps             | 174      |
|    iterations      | 1        |
|    time_elapsed    | 11       |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_l

KeyboardInterrupt: 

In [ ]:
pass

In [ ]:
import gymnasium as gym
import numpy as np
import time

gym.register(
    id="myoLegCustomWalk-v0",
    entry_point="myosuite.envs.myo.myobase.walk_v0:WalkEnvV0",
    max_episode_steps=1000,
    kwargs={
        "model_path": r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\models\80muscle\myoLeg80_HMEDI\myolegs_HMEDI.xml",
        "normalize_act": True,
        "min_height": 0.8,
        "max_rot": 0.8,
        "hip_period": 100,
        "reset_type": "init",
        "target_x_vel": 0.0,
        "target_y_vel": 1.2,
        "target_rot": None,
    },
)

def test_walk_environment():
    try:
        env = gym.make("myoLegCustomWalk-v0")
        obs, info = env.reset()
        
        total_reward = 0
        episode_length = 100
        
        for step in range(episode_length):
            action = env.action_space.sample()
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            total_reward += reward
            
            if step % 20 == 0:
                env.render()
            
            time.sleep(0.01)
            
            if done:
                break
        
        obs, info = env.reset()
        env.close()
        return True
        
    except Exception as e:
        return False

def quick_walk_test():
    try:
        env = gym.make("myoLegCustomWalk-v0")
        obs, info = env.reset()
        
        for i in range(10):
            action = env.action_space.sample()
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            if done:
                break
        
        env.close()
        return True
        
    except Exception as e:
        return False

if __name__ == "__main__":
    basic_success = quick_walk_test()
    
    if basic_success:
        test_walk_environment()

Custom environment 'myoLegCustomWalk-v0' registered successfully!
Model: 80-muscle leg model (HMEDI)
Task: Bipedal walking

Testing MyoLeg 80-Muscle Custom Walk Environment...
PATH DEBUGGING
Checking path: C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\models\80muscle\myoLeg80_HMEDI\myolegs_HMEDI.xml
Path exists: True
Is file: True
Parent directory exists: True
Files in parent directory:
  - assets
  - myolegs_HMEDI.xml

Alternative path format: C:/Users/rohan/Downloads/Exoskeleton_PD/exo_leg/myoassist/models/80muscle/myoLeg80_HMEDI/myolegs_HMEDI.xml
Alternative path exists: True

QUICK WALK TEST (NO RENDERING)
    MyoSuite: A contact-rich simulation suite for musculoskeletal motor control
        Vittorio Caggiano, Huawei Wang, Guillaume Durandau, Massimo Sartori, Vikash Kumar
        L4DC-2019 | https://sites.google.com/view/myosuite
    
Environment created successfully
Action space: (82,)
Observation space: (409,)
Number of muscles: 80
Step 1: reward=6.2470, done=False
S

c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\gymnasium\core.py:311: UserWarning: WARN: env.sim to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.sim` for environment variables or `env.get_wrapper_attr('sim')` that will search the reminding wrappers.
  logger.warn(


WALK ENVIRONMENT ANALYSIS
Environment ID: myoLegCustomWalk-v0
Action Space: Box(-1.0, 1.0, (82,), float32)
Action Space Shape: (82,)
Observation Space: Box(-10.0, 10.0, (409,), float32)
Observation Space Shape: (409,)

Model Structure:
Number of actuators (muscles): 80
Number of joints: 29
Number of bodies: 28

Muscle/Actuator names (first 15):
Error during walk test: 'MjModel' object has no attribute 'actuator_id2name'
Error type: AttributeError

❌ Unexpected error. Your model may be missing components that WalkEnv requires.

⚠️ Basic functionality works, but full test had issues.
Check that your model has the required joint and body names for walking.


In [ ]:
import gymnasium as gym
import numpy as np
import time

gym.register(
    id="myoLegCustomWalk-v0",
    entry_point="myosuite.envs.myo.myobase.walk_v0:WalkEnvV0",
    max_episode_steps=1000,
    kwargs={
        "model_path": r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\models\80muscle\myoLeg80_HMEDI\myolegs_HMEDI.xml",
        "normalize_act": True,
        "min_height": 0.8,
        "max_rot": 0.8,
        "hip_period": 100,
        "reset_type": "init",
        "target_x_vel": 0.0,
        "target_y_vel": 1.2,
        "target_rot": None,
    },
)

def test_walk_environment():
    try:
        env = gym.make("myoLegCustomWalk-v0")
        obs, info = env.reset()
        
        total_reward = 0
        episode_length = 100
        
        for step in range(episode_length):
            action = env.action_space.sample()
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            total_reward += reward
            
            if step % 20 == 0:
                env.render()
            
            time.sleep(0.01)
            
            if done:
                break
        
        obs, info = env.reset()
        env.close()
        return True
        
    except Exception as e:
        return False

def quick_walk_test():
    try:
        env = gym.make("myoLegCustomWalk-v0")
        obs, info = env.reset()
        
        for i in range(10):
            action = env.action_space.sample()
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            if done:
                break
        
        env.close()
        return True
        
    except Exception as e:
        return False

if __name__ == "__main__":
    basic_success = quick_walk_test()
    
    if basic_success:
        test_walk_environment()

In [ ]:
import numpy as np
from gymnasium import ObservationWrapper, ActionWrapper, spaces

class ObsAdapter(ObservationWrapper):
    def __init__(self, env, expected_obs_dim=403):
        super().__init__(env)
        self.expected_obs_dim = expected_obs_dim
        self.observation_space = spaces.Box(
            low=-10.0, high=10.0, shape=(expected_obs_dim,), dtype=np.float32
        )

    def observation(self, obs):
        if obs.shape[0] > self.expected_obs_dim:
            return obs[:self.expected_obs_dim]
        elif obs.shape[0] < self.expected_obs_dim:
            return np.pad(obs, (0, self.expected_obs_dim - obs.shape[0]))
        return obs


class ActionAdapter(ActionWrapper):
    def __init__(self, env, expected_action_dim=80):
        super().__init__(env)
        self.expected_action_dim = expected_action_dim
        self.action_space = spaces.Box(
            low=-1.0, high=1.0, shape=(expected_action_dim,), dtype=np.float32
        )

    def action(self, act):
        env_dim = self.env.action_space.shape[0]
        if len(act) > env_dim:
            return act[:env_dim]
        elif len(act) < env_dim:
            return np.pad(act, (0, env_dim - len(act)))
        return act

In [ ]:
from myosuite.utils import gym
from stable_baselines3 import PPO
import time

env = gym.make('myoLegCustomWalk-v0')

env = ObsAdapter(env, expected_obs_dim=403)
env = ActionAdapter(env, expected_action_dim=80)

obs, _ = env.reset()

model = PPO.load("LegWalk_policy", env, verbose=1)

done = False
tr = 0
env.mj_render()
time.sleep(3)

while not done:
    action, _ = model.predict(obs)
    obs, reward, done, _, info = env.step(action)
    env.mj_render()
    tr += reward

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\gymnasium\core.py:311: UserWarning: WARN: env.mj_render to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.mj_render` for environment variables or `env.get_wrapper_attr('mj_render')` that will search the reminding wrappers.
  logger.warn(


Final reward -20.710565880997734


In [ ]:
from stable_baselines3 import PPO
from myosuite.utils import gym

env = gym.make('myoLegCustomWalk-v0')

env = ObsAdapter(env, expected_obs_dim=403)
env = ActionAdapter(env, expected_action_dim=80)

log_dir = "./ppo_legwalk_tensorboard/"

model = PPO.load("LegWalk_hmedi_policy", env=env, verbose=1, tensorboard_log=log_dir)

model.learn(total_timesteps=1_000_000)

model.save('LegWalk_hmedi_policy')

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Starting policy learning
Logging to ./ppo_legwalk_tensorboard/PPO_4
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 156      |
|    ep_rew_mean     | 1.91e+03 |
| time/              |          |
|    fps             | 340      |
|    iterations      | 1        |
|    time_elapsed    | 6        |
|    total_timesteps | 2048     |
---------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 160        |
|    ep_rew_mean          | 1.92e+03   |
| time/                   |            |
|    fps                  | 316        |
|    iterations           | 2          |
|    time_elapsed         | 12         |
|    total_timesteps      | 4096       |
| train/                  |            |
|    approx_kl            | 0.34011686 |
|    clip_fraction        | 0.768      |
|    clip_range        

KeyboardInterrupt: 

In [ ]:
from myosuite.utils import gym
from stable_baselines3 import PPO
import time

env = gym.make('myoLegCustomWalk-v0')

env = ObsAdapter(env, expected_obs_dim=403)
env = ActionAdapter(env, expected_action_dim=80)

obs, _ = env.reset()

model = PPO.load("LegWalk_hmedi_policy", env, verbose=1)

done = False
tr = 0
env.mj_render()
time.sleep(3)

while not done:
    action, _ = model.predict(obs)
    obs, reward, done, _, info = env.step(action)
    env.mj_render()
    tr += reward

Exception ignored in: <function Renderer.__del__ at 0x00000260AA82CC20>
Traceback (most recent call last):
  File "c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\myosuite\renderer\renderer.py", line 142, in __del__
    self.close()
  File "c:\Users\rohan\anaconda3\envs\exo_s\Lib\site-packages\myosuite\renderer\mj_renderer.py", line 158, in close
    quit()
    ^^^^
NameError: name 'quit' is not defined


Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Final reward 1919.4086363425831


In [ ]:
#Exo only wrapper
import myosuite
from myosuite.utils import gym
import numpy as np
from stable_baselines3 import PPO
import gymnasium


class MyoFatiExoOnlyWrapper(gym.Env):

    
    def __init__(self, base_env, frozen_policy_path, fatigue_indices=None, fatigue_range=(0.7, 1.0), smart_reset=False):

        super(MyoFatiExoOnlyWrapper, self).__init__()
        
        self.base_env = base_env
        self.frozen_policy = PPO.load(frozen_policy_path)
        self.fatigue_indices = fatigue_indices
        self.fatigue_range = fatigue_range
        self.smart_reset = smart_reset
        
        self.full_action_dim = self.base_env.action_space.shape[0]
        self.exo_action_dim = 2  #
        self.muscle_action_dim = self.full_action_dim - self.exo_action_dim
        
        try:
            self.n_muscles = len(self.base_env.unwrapped.muscle_fatigue.MA)
        except AttributeError:
            self.n_muscles = self.muscle_action_dim
            
        self.action_space = gym.spaces.Box(
            low=self.base_env.action_space.low[-self.exo_action_dim:],
            high=self.base_env.action_space.high[-self.exo_action_dim:],
            dtype=np.float32
        )
        
        self.observation_space = self.base_env.observation_space
        
        if self.frozen_policy.observation_space.shape[0] != self.base_env.observation_space.shape[0]:
            print(f"Warning: Frozen policy obs dim ({self.frozen_policy.observation_space.shape[0]}) "
                  f"!= base env obs dim ({self.base_env.observation_space.shape[0]})")
        
        if self.frozen_policy.action_space.shape[0] != self.muscle_action_dim:
            raise ValueError(
                f"Frozen policy action dim ({self.frozen_policy.action_space.shape[0]}) "
                f"!= muscle action dim ({self.muscle_action_dim})"
            )
        
        print(f"  - Full action space: {self.full_action_dim} (muscle: {self.muscle_action_dim}, exo: {self.exo_action_dim})")
        print(f"  - Number of muscles: {self.n_muscles}")
        print(f"  - Fatigue indices: {self.fatigue_indices}")
        print(f"  - Fatigue range: {self.fatigue_range}")
        print(f"  - Smart reset: {self.smart_reset}")
        print(f"  - Wrapper action space: {self.action_space.shape} (exo only)")
        print(f"  - Observation space: {self.observation_space.shape}")
        
        # Set up fatigue reset mode
        if not self.smart_reset:
            try:
                self.base_env.unwrapped.set_fatigue_reset_random(True)
            except AttributeError:
                print("h")

    def _apply_selective_fatigue(self):

            
        muscle_fatigue = self.base_env.unwrapped.muscle_fatigue
        
        if self.fatigue_indices is not None:
            muscle_fatigue.MF[:] = 0.0
            muscle_fatigue.MA[:] = 0.0
            muscle_fatigue.MR[:] = 1.0
            
            for idx in self.fatigue_indices:
                if idx < len(muscle_fatigue.MF):
                    fatigue_level = np.random.uniform(
                        1.0 - self.fatigue_range[1],  
                        1.0 - self.fatigue_range[0]
                    )
                    muscle_fatigue.MF[idx] = fatigue_level
                    
                    remaining = 1.0 - fatigue_level
                    split = np.random.uniform(0.0, 1.0)
                    muscle_fatigue.MA[idx] = remaining * split
                    muscle_fatigue.MR[idx] = remaining * (1.0 - split)
                    
            print(f"Applied fatigue to muscles {self.fatigue_indices}: "
                  f"MF range [{muscle_fatigue.MF[self.fatigue_indices].min():.3f}, "
                  f"{muscle_fatigue.MF[self.fatigue_indices].max():.3f}]")
        else:
            for i in range(len(muscle_fatigue.MF)):
                fatigue_level = np.random.uniform(
                    1.0 - self.fatigue_range[1],
                    1.0 - self.fatigue_range[0]
                )
                muscle_fatigue.MF[i] = fatigue_level
                
                remaining = 1.0 - fatigue_level
                split = np.random.uniform(0.0, 1.0)
                muscle_fatigue.MA[i] = remaining * split
                muscle_fatigue.MR[i] = remaining * (1.0 - split)

    def reset(self, **kwargs):
        obs, info = self.base_env.reset(**kwargs)
        
        if self.smart_reset:
            self._apply_selective_fatigue()
        
        return obs, info

    def step(self, exo_action):
    
        exo_action = np.array(exo_action, dtype=np.float32).reshape(-1)
        
        if len(exo_action) != self.exo_action_dim:
            raise ValueError(f"Expected {self.exo_action_dim} exoskeleton actions, got {len(exo_action)}")
        
        try:
            current_obs = self.base_env._get_obs()
        except AttributeError:
            current_obs = self.base_env.unwrapped.get_obs()
        
        muscle_actions, _ = self.frozen_policy.predict(current_obs, deterministic=True)
        
        muscle_actions = np.array(muscle_actions, dtype=np.float32).reshape(-1)
        
        if len(muscle_actions) != self.muscle_action_dim:
            raise ValueError(f"Frozen policy produced {len(muscle_actions)} actions, "
                           f"expected {self.muscle_action_dim}")
        
        full_action = np.concatenate([muscle_actions, exo_action])
        
        obs, reward, done, truncated, info = self.base_env.step(full_action)
        
        return obs, reward, done, truncated, info

    def render(self, *args, **kwargs):
        """Render the environment"""
        return self.base_env.render(*args, **kwargs)

    def close(self):
        """Close the environment"""
        self.base_env.close()
    
    @property
    def muscle_fatigue_levels(self):
        try:
            return self.base_env.unwrapped.muscle_fatigue.MF.copy()
        except AttributeError:
            return np.zeros(self.n_muscles)
    
    @property
    def muscle_active_levels(self):
        try:
            return self.base_env.unwrapped.muscle_fatigue.MA.copy()
        except AttributeError:
            return np.zeros(self.n_muscles)
    
    @property
    def muscle_resting_levels(self):
        try:
            return self.base_env.unwrapped.muscle_fatigue.MR.copy()
        except AttributeError:
            return np.ones(self.n_muscles)




In [ ]:
from myosuite.utils import gym
from stable_baselines3 import PPO
import time
    
base_env = gym.make('myoFatiLegWalkCustom-v0')
    
#wrapped_env = MyoFatiExoOnlyWrapper(
#        base_env=base_env,
#        frozen_policy_path='path/to/trained_policy.zip',
#        fatigue_indices=[0, 4, 11, 15],  # Only these muscles get fatigued
#        fatigue_range=(0.7, 1.0),        # 70-100% strength for fatigued muscles
#        smart_reset=True
#    )
env = ObsAdapter(base_env, expected_obs_dim=403)
env = ActionAdapter(env, expected_action_dim=80)

obs, _ = env.reset()

model = PPO.load("LegWalk_hmedi_policy", env, verbose=1)

MF = np.random.uniform(0.999, 1.0, size=env.env.base_env.n_muscles)

remaining = 1.0 - MF
split = np.random.uniform(0.0, 1.0, size=env.env.base_env.n_muscles)  # split ratio
MA = remaining * split
MR = remaining * (1.0 - split)

env.env.base_env.unwrapped.muscle_fatigue.MA[:] = MA
env.env.base_env.unwrapped.muscle_fatigue.MR[:] = MR
env.env.base_env.unwrapped.muscle_fatigue.MF[:] = MF

done = False
tr = 0
env.mj_render()
time.sleep(3)

while not done:
    action, _ = model.predict(obs)
    obs, reward, done, _, info = env.step(action)
    env.mj_render()
    tr += reward

print("Final reward", tr)


In [ ]:
import myosuite
from myosuite.utils import gym
import numpy as np
from stable_baselines3 import PPO
import gymnasium


class MyoFatiExoOnlyWrapper(gym.Env):
    
    def __init__(self, base_env, frozen_policy_path, fatigue_indices=None, fatigue_range=(0.7, 1.0), smart_reset=False):
        super(MyoFatiExoOnlyWrapper, self).__init__()
        
        self.base_env = base_env
        self.frozen_policy = PPO.load(frozen_policy_path)
        self.fatigue_indices = fatigue_indices
        self.fatigue_range = fatigue_range
        self.smart_reset = smart_reset
        
        self.full_action_dim = self.base_env.action_space.shape[0]
        self.exo_action_dim = 2
        self.muscle_action_dim = self.full_action_dim - self.exo_action_dim
        
        try:
            self.n_muscles = len(self.base_env.unwrapped.muscle_fatigue.MA)
        except AttributeError:
            self.n_muscles = self.muscle_action_dim
            
        self.action_space = gym.spaces.Box(
            low=self.base_env.action_space.low[-self.exo_action_dim:],
            high=self.base_env.action_space.high[-self.exo_action_dim:],
            dtype=np.float32
        )
        
        self.observation_space = self.base_env.observation_space
        
        if self.frozen_policy.observation_space.shape[0] != self.base_env.observation_space.shape[0]:
            pass
        
        if self.frozen_policy.action_space.shape[0] != self.muscle_action_dim:
            raise ValueError(
                f"Frozen policy action dim ({self.frozen_policy.action_space.shape[0]}) "
                f"!= muscle action dim ({self.muscle_action_dim})"
            )
        
        if not self.smart_reset:
            try:
                self.base_env.unwrapped.set_fatigue_reset_random(True)
            except AttributeError:
                pass

    def _apply_selective_fatigue(self):
        if not hasattr(self.base_env.unwrapped, 'muscle_fatigue'):
            return
            
        muscle_fatigue = self.base_env.unwrapped.muscle_fatigue
        
        if self.fatigue_indices is not None:
            muscle_fatigue.MF[:] = 0.0
            muscle_fatigue.MA[:] = 0.0
            muscle_fatigue.MR[:] = 1.0
            
            for idx in self.fatigue_indices:
                if idx < len(muscle_fatigue.MF):
                    fatigue_level = np.random.uniform(
                        1.0 - self.fatigue_range[1],
                        1.0 - self.fatigue_range[0]
                    )
                    muscle_fatigue.MF[idx] = fatigue_level
                    
                    remaining = 1.0 - fatigue_level
                    split = np.random.uniform(0.0, 1.0)
                    muscle_fatigue.MA[idx] = remaining * split
                    muscle_fatigue.MR[idx] = remaining * (1.0 - split)
        else:
            for i in range(len(muscle_fatigue.MF)):
                fatigue_level = np.random.uniform(
                    1.0 - self.fatigue_range[1],
                    1.0 - self.fatigue_range[0]
                )
                muscle_fatigue.MF[i] = fatigue_level
                
                remaining = 1.0 - fatigue_level
                split = np.random.uniform(0.0, 1.0)
                muscle_fatigue.MA[i] = remaining * split
                muscle_fatigue.MR[i] = remaining * (1.0 - split)

    def reset(self, **kwargs):
        obs, info = self.base_env.reset(**kwargs)
        
        if self.smart_reset:
            self._apply_selective_fatigue()
        
        return obs, info

    def step(self, exo_action):
        exo_action = np.array(exo_action, dtype=np.float32).reshape(-1)
        
        if len(exo_action) != self.exo_action_dim:
            raise ValueError(f"Expected {self.exo_action_dim} exoskeleton actions, got {len(exo_action)}")
        
        try:
            current_obs = self.base_env._get_obs()
        except AttributeError:
            current_obs = self.base_env.unwrapped.get_obs()
        
        muscle_actions, _ = self.frozen_policy.predict(current_obs, deterministic=True)
        
        muscle_actions = np.array(muscle_actions, dtype=np.float32).reshape(-1)
        
        if len(muscle_actions) != self.muscle_action_dim:
            raise ValueError(f"Frozen policy produced {len(muscle_actions)} actions, "
                           f"expected {self.muscle_action_dim}")
        
        full_action = np.concatenate([muscle_actions, exo_action])
        
        obs, reward, done, truncated, info = self.base_env.step(full_action)
        
        return obs, reward, done, truncated, info

    def render(self, *args, **kwargs):
        return self.base_env.render(*args, **kwargs)

    def close(self):
        self.base_env.close()
    
    @property
    def muscle_fatigue_levels(self):
        try:
            return self.base_env.unwrapped.muscle_fatigue.MF.copy()
        except AttributeError:
            return np.zeros(self.n_muscles)
    
    @property
    def muscle_active_levels(self):
        try:
            return self.base_env.unwrapped.muscle_fatigue.MA.copy()
        except AttributeError:
            return np.zeros(self.n_muscles)
    
    @property
    def muscle_resting_levels(self):
        try:
            return self.base_env.unwrapped.muscle_fatigue.MR.copy()
        except AttributeError:
            return np.ones(self.n_muscles)

  Cloning https://github.com/aravindr93/mjrl.git to c:\users\rohan\appdata\local\temp\pip-req-build-u7oefe3d
  Resolved https://github.com/aravindr93/mjrl.git to commit 3871d93763d3b49c4741e6daeaebbc605fe140dc
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for mjrl: filename=mjrl-1.0.0-py3-none-any.whl size=62521 sha256=c2d8cd17f56f5c90a3c0ab2fa63570d9c4d8d5130059854b464cbb3c9b5816a7
  Stored in directory: C:\Users\rohan\AppData\Local\Temp\pip-ephem-wheel-cache-y2vu92n0\wheels\17\19\07\51f2dad17a1ce98f641a348271d8f6adfa3bbe05886b0ce2d8
Successfully built mjrl


  Running command git clone --filter=blob:none --quiet https://github.com/aravindr93/mjrl.git 'C:\Users\rohan\AppData\Local\Temp\pip-req-build-u7oefe3d'
  DEPRECATION: Building 'mjrl' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'mjrl'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [ ]:
#import ReflexCtrInterface
import numpy as np
import skvideo.io

from base64 import b64encode
from IPython.display import HTML

from myosuite.utils.quat_math import quat2euler

sim_time = 5
dt = 0.01
steps = int(sim_time/dt)
frames = []

params = np.loadtxt('baseline_params.txt')

Myo_env = MyoLegReflex()
Myo_env.reset()

Myo_env.set_control_params(params)

for timestep in range(steps):
    frame = Myo_env.env.sim.renderer.render_offscreen(camera_id=1)
    frames.append(frame)
    _, isDone, _, _ = Myo_env.run_reflex_step()

    if isDone:
        break
    
skvideo.io.vwrite("MyoReflex_output.mp4", 
                  np.asarray(frames),inputdict={"-r":"100"}, outputdict={"-r" : "100", "-pix_fmt": "yuv420p"})

show_video("MyoReflex_output.mp4")

Seed added -  0
Stopped at - 88


In [ ]:
from __future__ import division
import numpy as np

import myosuite
from myosuite.utils import gym

import numpy as np
import os

from myosuite.utils.quat_math import quat2euler
from myosuite.utils.quat_math import euler2quat

class MyoLegReflex(object):

    DEFAULT_INIT_POSE = {}
    DEFAULT_INIT_POSE['model_pose'] = {'yaw':np.deg2rad(0),'pitch':np.deg2rad(15),'roll':np.deg2rad(0)}
    DEFAULT_INIT_POSE['model_height'] = 0.92
    DEFAULT_INIT_POSE['joint_angles'] = {
        'hip_flexion_r': np.deg2rad(180-190),
        'hip_flexion_l': np.deg2rad(180-155),
        'knee_angle_r': np.deg2rad(180-165),
        'knee_angle_l': np.deg2rad(180-180),
        'ankle_angle_r': np.deg2rad(90-90),
        'ankle_angle_l': np.deg2rad(90-100),
    }
    DEFAULT_INIT_POSE['velocity'] = {'cartesian':[1.5, 0.0, 0.0]}

    def __init__(self, init_dict=DEFAULT_INIT_POSE, dt=0.01, mode='3D', sim_time=2.0, seed=0):
        self.dt = dt
        self.t = 0
        self.mode = mode
        
        self.n_par = len(LocoCtrl.cp_keys)
        control_dimension = 3
        
        self.cp_map = LocoCtrl.cp_map
        self.ReflexCtrl = LocoCtrl(self.dt, control_dimension=control_dimension, params=np.ones(self.n_par))

        self.sim_time = sim_time
        self.timestep_limit = int(self.sim_time/self.dt)

        self.init_dict = init_dict
        self.seed = seed

        self.env = gym.make('myoLegStandRandom-v0', normalize_act=False)

        self.env.reset()
        self.env.seed(seed)

        self.muscle_labels = {}
        self.muscles_dict = {}
        self.muscle_Fmax = {}
        self.muscle_L0 = {}

        self.init_pelvis = np.zeros(3,)

        self.footstep = {}
        self.footstep['n'] = 0
        self.footstep['new'] = False
        self.footstep['r_contact'] = 0
        self.footstep['l_contact'] = 0
        
        self.cp = self.ReflexCtrl.cp

    def reset(self):
        
        self.env.reset()
        self.env.seed(self.seed)
        
        self.ReflexCtrl.reset()
        
        self._set_muscle_groups()
        self._set_initial_pose(self.init_dict)

        
    def update(self):
        self.t += self.dt
        self.ReflexCtrl.update(self.get_obs_dict())
        return self.ReflexCtrl.stim.copy()
        

    def set_control_params(self, params):
        self.ReflexCtrl.set_control_params(params)

    def set_control_params_RL(self, s_leg, params):
        self.ReflexCtrl.set_control_params_RL(s_leg, params)

    def get_obs_dict(self):
        pel_euler = quat2euler(self.env.sim.data.body('pelvis').xquat.copy())
        pelvis_roll = pel_euler[0] - (np.pi/2)
        pelvis_pitch = pel_euler[2] * (-1)
        pelvis_yaw = pel_euler[1] * (-1)

        temp_seg_vel = self.env.sim.data.object_velocity('pelvis','body', local_frame=False).copy()
        lin_seg_vel = temp_seg_vel[0]
        dx_local, dy_local = self.rotate_frame(lin_seg_vel[0], lin_seg_vel[1], pelvis_yaw)
        pelvis_vel = np.hstack((np.array([dx_local, dy_local, lin_seg_vel[2]]),
                                            temp_seg_vel[1] ))

        temp_right = (self.env.sim.data.sensor('r_foot').data[0].copy() + self.env.sim.data.sensor('r_toes').data[0].copy())
        temp_left = (self.env.sim.data.sensor('l_foot').data[0].copy() + self.env.sim.data.sensor('l_toes').data[0].copy())

        sensor_data = {'body':{}, 'r_leg':{}, 'l_leg':{}}
        sensor_data['body']['theta'] = [pelvis_roll,
                                        pelvis_pitch]

        sensor_data['body']['d_pos'] = [pelvis_vel[0],
                                        pelvis_vel[1]]
        
        sensor_data['body']['dtheta'] = [pelvis_vel[3],
                                        pelvis_vel[4]]
        
        sensor_data['r_leg']['load_ipsi'] = temp_right / (np.sum(self.env.sim.model.body_mass)*9.8)
        sensor_data['l_leg']['load_ipsi'] = temp_left / (np.sum(self.env.sim.model.body_mass)*9.8)

        for s_leg, s_legc in zip(['r_leg', 'l_leg'], ['l_leg', 'r_leg']):

            sensor_data[s_leg]['contact_ipsi'] = 1 if sensor_data[s_leg]['load_ipsi'] > 0.1 else 0
            sensor_data[s_leg]['contact_contra'] = 1 if sensor_data[s_legc]['load_ipsi'] > 0.1 else 0
            sensor_data[s_leg]['load_contra'] = sensor_data[s_legc]['load_ipsi']

            sensor_data[s_leg]['phi_hip'] = (np.pi - self.env.sim.data.jnt(f"hip_flexion_{s_leg[0]}").qpos[0].copy())
            sensor_data[s_leg]['phi_knee'] = (np.pi - self.env.sim.data.jnt(f"knee_angle_{s_leg[0]}").qpos[0].copy())
            sensor_data[s_leg]['phi_ankle'] = (0.5*np.pi - self.env.sim.data.jnt(f"ankle_angle_{s_leg[0]}").qpos[0].copy())
            sensor_data[s_leg]['dphi_knee'] = self.env.sim.data.jnt(f"knee_angle_{s_leg[0]}").qvel[0].copy()

            sensor_data[s_leg]['alpha'] = sensor_data[s_leg]['phi_hip'] - 0.5*sensor_data[s_leg]['phi_knee']
            dphi_hip = self.env.sim.data.jnt(f"hip_flexion_{s_leg[0]}").qvel[0].copy()
            sensor_data[s_leg]['dalpha'] = dphi_hip - 0.5*sensor_data[s_leg]['dphi_knee']

            sensor_data[s_leg]['alpha_f'] = (-1*self.env.sim.data.jnt(f"hip_adduction_{s_leg[0]}").qpos[0].copy()) + 0.5*np.pi

            temp_mus_force = self.env.sim.data.actuator_force.copy()

            sensor_data[s_leg]['F_RF'] = -1*np.mean( temp_mus_force[self.muscles_dict[s_leg]['RF']] / (self.muscle_Fmax[s_leg]['RF']) )
            sensor_data[s_leg]['F_VAS'] = -1*np.mean( temp_mus_force[self.muscles_dict[s_leg]['VAS']] / (self.muscle_Fmax[s_leg]['VAS']) )
            sensor_data[s_leg]['F_GAS'] = -1*np.mean( temp_mus_force[self.muscles_dict[s_leg]['GAS']] / (self.muscle_Fmax[s_leg]['GAS']) )
            sensor_data[s_leg]['F_SOL'] = -1*np.mean( temp_mus_force[self.muscles_dict[s_leg]['SOL']] / (self.muscle_Fmax[s_leg]['SOL']) )

        return sensor_data

    def run_reflex_step(self):
        is_done = False

        new_act = self.reflex2mujoco(self.update())
        self.env.step(new_act)

        self.update_footstep()
        
        out_dict = self.get_obs_dict()
        
        temp_pel_euler = quat2euler(self.env.sim.data.body('root').xquat.copy())
        
        if self.env.sim.data.body('pelvis').xpos[2] < 0.65:
            is_done = True
        if temp_pel_euler[1] < np.deg2rad(-30) or temp_pel_euler[1] > np.deg2rad(30):
            is_done = True
        
        return [ out_dict, is_done, np.round(self.env.sim.data.time,2), new_act]

    def _set_muscle_groups(self):
        glu_r = [self.env.sim.model.actuator('glmax1_r').id,
        self.env.sim.model.actuator('glmax2_r').id,
        self.env.sim.model.actuator('glmax3_r').id,
        self.env.sim.model.actuator('glmed3_r').id]

        glu_l = [self.env.sim.model.actuator('glmax1_l').id,
        self.env.sim.model.actuator('glmax2_l').id,
        self.env.sim.model.actuator('glmax3_l').id,
        self.env.sim.model.actuator('glmed3_l').id]

        glu_r_lbl = ['glmax1_r','glmax2_r','glmax3_r','glmed3_r']
        glu_l_lbl = ['glmax1_l','glmax2_l','glmax3_l','glmed3_l']

        ham_r = [self.env.sim.model.actuator('semimem_r').id,
                self.env.sim.model.actuator('semiten_r').id,
                self.env.sim.model.actuator('bflh_r').id]

        ham_l = [self.env.sim.model.actuator('semimem_l').id,
                self.env.sim.model.actuator('semiten_l').id,
                self.env.sim.model.actuator('bflh_l').id]

        ham_r_lbl = ['semimem_r','semiten_r','bflh_r']
        ham_l_lbl = ['semimem_l','semiten_l','bflh_l']

        bfsh_r = [self.env.sim.model.actuator('bfsh_r').id]

        bfsh_l = [self.env.sim.model.actuator('bfsh_l').id]

        bfsh_r_lbl = ['bfsh_r']
        bfsh_l_lbl = ['bfsh_l']

        gas_r = [self.env.sim.model.actuator('gaslat_r').id,
                self.env.sim.model.actuator('gasmed_r').id]

        gas_l = [self.env.sim.model.actuator('gaslat_l').id,
                self.env.sim.model.actuator('gasmed_l').id]

        gas_r_lbl = ['gaslat_r','gasmed_r']
        gas_l_lbl = ['gaslat_l','gasmed_l']

        sol_r = [self.env.sim.model.actuator('soleus_r').id,
                self.env.sim.model.actuator('perbrev_r').id,
                self.env.sim.model.actuator('perlong_r').id,
                self.env.sim.model.actuator('tibpost_r').id]

        sol_l = [self.env.sim.model.actuator('soleus_l').id,
                self.env.sim.model.actuator('perbrev_l').id,
                self.env.sim.model.actuator('perlong_l').id,
                self.env.sim.model.actuator('tibpost_l').id]

        sol_r_lbl = ['soleus_r','perbrev_r','perlong_r','tibpost_r']
        sol_l_lbl = ['soleus_l','perbrev_l','perlong_l','tibpost_l']

        hfl_r = [self.env.sim.model.actuator('psoas_r').id,
                self.env.sim.model.actuator('iliacus_r').id]

        hfl_l = [self.env.sim.model.actuator('psoas_l').id,
                self.env.sim.model.actuator('iliacus_l').id]

        hfl_r_lbl = ['psoas_r','iliacus_r']
        hfl_l_lbl = ['psoas_l','iliacus_l']

        hab_r = [self.env.sim.model.actuator('piri_r').id,
        self.env.sim.model.actuator('sart_r').id,
        self.env.sim.model.actuator('glmed1_r').id,
        self.env.sim.model.actuator('glmed2_r').id,
        self.env.sim.model.actuator('glmin1_r').id,
        self.env.sim.model.actuator('glmin2_r').id,
        self.env.sim.model.actuator('glmin3_r').id]

        hab_l = [self.env.sim.model.actuator('piri_l').id,
        self.env.sim.model.actuator('sart_l').id,
        self.env.sim.model.actuator('glmed1_l').id,
        self.env.sim.model.actuator('glmed2_l').id,
        self.env.sim.model.actuator('glmin1_l').id,
        self.env.sim.model.actuator('glmin2_l').id,
        self.env.sim.model.actuator('glmin3_l').id]

        hab_r_lbl = ['piri_r','sart_r','glmed1_r','glmed2_r','glmin1_r','glmin2_r','glmin3_r']
        hab_l_lbl = ['piri_l','sart_l','glmed1_l','glmed2_l','glmin1_l','glmin2_l','glmin3_l']

        had_r = [self.env.sim.model.actuator('addbrev_r').id,
        self.env.sim.model.actuator('addlong_r').id,
        self.env.sim.model.actuator('addmagDist_r').id,
        self.env.sim.model.actuator('addmagIsch_r').id,
        self.env.sim.model.actuator('addmagMid_r').id,
        self.env.sim.model.actuator('addmagProx_r').id,
        self.env.sim.model.actuator('grac_r').id]

        had_l = [self.env.sim.model.actuator('addbrev_l').id,
        self.env.sim.model.actuator('addlong_l').id,
        self.env.sim.model.actuator('addmagDist_l').id,
        self.env.sim.model.actuator('addmagIsch_l').id,
        self.env.sim.model.actuator('addmagMid_l').id,
        self.env.sim.model.actuator('addmagProx_l').id,
        self.env.sim.model.actuator('grac_l').id]

        had_r_lbl = ['addbrev_r','addlong_r','addmagDist_r','addmagIsch_r','addmagMid_r','addmagProx_r','grac_r']
        had_l_lbl = ['addbrev_l','addlong_l','addmagDist_l','addmagIsch_l','addmagMid_l','addmagProx_l','grac_l']

        rf_r = [self.env.sim.model.actuator('recfem_r').id]

        rf_l = [self.env.sim.model.actuator('recfem_l').id]

        rf_r_lbl = ['recfem_r']
        rf_l_lbl = ['recfem_l']

        vas_r = [self.env.sim.model.actuator('vasint_r').id,
        self.env.sim.model.actuator('vaslat_r').id,
        self.env.sim.model.actuator('vasmed_r').id]

        vas_l = [self.env.sim.model.actuator('vasint_l').id,
        self.env.sim.model.actuator('vaslat_l').id,
        self.env.sim.model.actuator('vasmed_l').id]

        vas_r_lbl = ['vasint_r','vaslat_r','vasmed_r']
        vas_l_lbl = ['vasint_l','vaslat_l','vasmed_l']

        ta_r = [self.env.sim.model.actuator('tibant_r').id]

        ta_l = [self.env.sim.model.actuator('tibant_l').id]

        ta_r_lbl = ['tibant_r']
        ta_l_lbl = ['tibant_l']

        self.muscles_dict['r_leg'] = {}
        self.muscles_dict['r_leg']['HAB'] = hab_r
        self.muscles_dict['r_leg']['HAD'] = had_r
        self.muscles_dict['r_leg']['GLU'] = glu_r
        self.muscles_dict['r_leg']['HAM'] = ham_r
        self.muscles_dict['r_leg']['BFSH'] = bfsh_r
        self.muscles_dict['r_leg']['GAS'] = gas_r
        self.muscles_dict['r_leg']['SOL'] = sol_r
        self.muscles_dict['r_leg']['HFL'] = hfl_r
        self.muscles_dict['r_leg']['RF'] = rf_r
        self.muscles_dict['r_leg']['VAS'] = vas_r
        self.muscles_dict['r_leg']['TA'] = ta_r

        self.muscles_dict['l_leg'] = {}
        self.muscles_dict['l_leg']['HAB'] = hab_l
        self.muscles_dict['l_leg']['HAD'] = had_l
        self.muscles_dict['l_leg']['GLU'] = glu_l
        self.muscles_dict['l_leg']['HAM'] = ham_l
        self.muscles_dict['l_leg']['BFSH'] = bfsh_l
        self.muscles_dict['l_leg']['GAS'] = gas_l
        self.muscles_dict['l_leg']['SOL'] = sol_l
        self.muscles_dict['l_leg']['HFL'] = hfl_l
        self.muscles_dict['l_leg']['RF'] = rf_l
        self.muscles_dict['l_leg']['VAS'] = vas_l
        self.muscles_dict['l_leg']['TA'] = ta_l

        self.muscle_labels['r_leg'] = {}
        self.muscle_labels['r_leg']['HAB'] = hab_r_lbl
        self.muscle_labels['r_leg']['HAD'] = had_r_lbl
        self.muscle_labels['r_leg']['GLU'] = glu_r_lbl
        self.muscle_labels['r_leg']['HAM'] = ham_r_lbl
        self.muscle_labels['r_leg']['BFSH'] = bfsh_r_lbl
        self.muscle_labels['r_leg']['GAS'] = gas_r_lbl
        self.muscle_labels['r_leg']['SOL'] = sol_r_lbl
        self.muscle_labels['r_leg']['HFL'] = hfl_r_lbl
        self.muscle_labels['r_leg']['RF'] = rf_r_lbl
        self.muscle_labels['r_leg']['VAS'] = vas_r_lbl
        self.muscle_labels['r_leg']['TA'] = ta_r_lbl

        self.muscle_labels['l_leg'] = {}
        self.muscle_labels['l_leg']['HAB'] = hab_l_lbl
        self.muscle_labels['l_leg']['HAD'] = had_l_lbl
        self.muscle_labels['l_leg']['GLU'] = glu_l_lbl
        self.muscle_labels['l_leg']['HAM'] = ham_l_lbl
        self.muscle_labels['l_leg']['BFSH'] = bfsh_l_lbl
        self.muscle_labels['l_leg']['GAS'] = gas_l_lbl
        self.muscle_labels['l_leg']['SOL'] = sol_l_lbl
        self.muscle_labels['l_leg']['HFL'] = hfl_l_lbl
        self.muscle_labels['l_leg']['RF'] = rf_l_lbl
        self.muscle_labels['l_leg']['VAS'] = vas_l_lbl
        self.muscle_labels['l_leg']['TA'] = ta_l_lbl

        temp_L0 = (self.env.sim.model.actuator_lengthrange[:,0] - self.env.sim.model.tendon_lengthspring[:,0]) / self.env.sim.model.actuator_biasprm[:,0]

        for x in self.muscles_dict:
            self.muscle_Fmax[x] = {}
            self.muscle_L0[x] = {}
            for y in self.muscles_dict[x]:
                self.muscle_Fmax[x][y] = self.env.sim.model.actuator_biasprm[self.muscles_dict[x][y], 2].copy()
                self.muscle_L0[x][y] = temp_L0[self.muscles_dict[x][y]]


    def _set_initial_pose(self, init_dict):
        self.init_pelvis = self.env.sim.data.body('pelvis').xpos.copy()

        temp_quat_util = euler2quat([init_dict['model_pose']['roll'], 
                                         init_dict['model_pose']['pitch'], 
                                         init_dict['model_pose']['yaw']])

        self.env.sim.data.qpos[3] = temp_quat_util[0]
        self.env.sim.data.qpos[4] = temp_quat_util[1]
        self.env.sim.data.qpos[5] = temp_quat_util[2]
        self.env.sim.data.qpos[6] = temp_quat_util[3]

        self.env.sim.data.qvel[0] = init_dict['velocity']['cartesian'][0]
        self.env.sim.data.qvel[1] = init_dict['velocity']['cartesian'][1]
        self.env.sim.data.qvel[2] = init_dict['velocity']['cartesian'][2]

        for joint_name in init_dict['joint_angles'].keys():
            self.env.sim.data.joint(joint_name).qpos[0] = init_dict['joint_angles'][joint_name]
        
        if 'height_offset' in init_dict.keys():
            height_offset = init_dict['height_offset']
        else:
            height_offset = 0

        self.env.sim.data.qpos[0] = 0
        self.env.sim.data.qpos[1] = 0
        self.env.sim.data.qpos[2] =  init_dict['model_height'] + height_offset
        
        self.env.sim.forward()

    def update_footstep(self):
        
        r_contact = True if (self.env.sim.data.sensor('r_foot').data[0].copy()) > 0.1*(np.sum(self.env.sim.model.body_mass)*9.8) else False
        l_contact = True if (self.env.sim.data.sensor('l_foot').data[0].copy()) > 0.1*(np.sum(self.env.sim.model.body_mass)*9.8) else False

        self.footstep['new'] = False
        if ( (not self.footstep['r_contact'] and r_contact) or (not self.footstep['l_contact'] and l_contact) ):
            self.footstep['new'] = True
            self.footstep['n'] += 1

        self.footstep['r_contact'] = r_contact
        self.footstep['l_contact'] = l_contact

    
    def reflex2mujoco(self, output):
        
        mus_act = np.zeros((80,))
        mus_act[:] = 0

        legs = ['r_leg', 'l_leg']
        musc_idx = self.muscles_dict['r_leg'].keys()

        for s_leg in legs:
            for musc in musc_idx:
                mus_act[self.muscles_dict[s_leg][musc]] = output[s_leg][musc]
        
        return mus_act

    def rotate_frame(self, x, y, theta):
        x_rot = np.cos(theta)*x - np.sin(theta)*y
        y_rot = np.sin(theta)*x + np.cos(theta)*y
        return x_rot, y_rot

In [ ]:
from __future__ import division
import numpy as np

class LocoCtrl(object):
    DEBUG = 0

    RIGHT = 0
    LEFT = 1

    m_keys = ['HAB', 'HAD', 'HFL', 'GLU', 'HAM', 'RF', 'VAS', 'BFSH', 'GAS', 'SOL', 'TA']
    s_b_keys = ['theta', 'd_pos', 'dtheta']
    s_l_keys = [
        'contact_ipsi', 'contact_contra', 'load_ipsi', 'load_contra',
        'alpha', 'alpha_f', 'dalpha',
        'phi_hip', 'phi_knee', 'phi_ankle', 'dphi_knee'
        'F_RF', 'F_VAS', 'F_GAS', 'F_SOL',
        ]
    cs_keys = [
        'ph_st',
        'ph_st_csw',
        'ph_st_sw0',
        'ph_sw',
        'ph_sw_flex_k',
        'ph_sw_hold_k',
        'ph_sw_stop_l',
        'ph_sw_hold_l'
        ]
    cp_keys = [
        'theta_tgt', 'c0', 'cv', 'alpha_delta',
        'knee_sw_tgt', 'knee_tgt', 'knee_off_st', 'ankle_tgt',
        'HFL_3_PG', 'HFL_3_DG', 'HFL_6_PG', 'HFL_6_DG', 'HFL_10_PG',
        'GLU_3_PG', 'GLU_3_DG', 'GLU_6_PG', 'GLU_6_DG', 'GLU_10_PG',
        'HAM_3_GLU', 'HAM_9_PG',
        'RF_1_FG', 'RF_8_DG_knee',
        'VAS_1_FG', 'VAS_2_PG', 'VAS_10_PG',
        'BFSH_2_PG', 'BFSH_7_DG_alpha', 'BFSH_7_PG', 'BFSH_8_DG', 'BFSH_8_PG',
        'BFSH_9_G_HAM', 'BFSH_9_HAM0', 'BFSH_10_PG',
        'GAS_2_FG',
        'SOL_1_FG',
        'TA_5_PG', 'TA_5_G_SOL',
        'theta_tgt_f', 'c0_f', 'cv_f',
        'HAB_3_PG', 'HAB_3_DG', 'HAB_6_PG',
        'HAD_3_PG', 'HAD_3_DG', 'HAD_6_PG'
        ]

    m_map = dict(zip(m_keys, range(len(m_keys))))
    s_b_map = dict(zip(s_b_keys, range(len(s_b_keys))))
    s_l_map = dict(zip(s_l_keys, range(len(s_l_keys))))
    cs_map = dict(zip(cs_keys, range(len(cs_keys))))
    cp_map = dict(zip(cp_keys, range(len(cp_keys))))

    def __init__(self, TIMESTEP, control_mode=1, control_dimension=3, params=np.ones(len(cp_keys))):
        if self.DEBUG:
            pass

        self.control_mode = control_mode
        self.control_dimension = control_dimension

        if self.control_mode == 0:
            self.brain_control_on = 0
        elif self.control_mode == 1:
            self.brain_control_on = 1

        self.spinal_control_phase = {}
        self.in_contact = {}
        self.brain_command = {}
        self.stim = {}

        self.n_par = len(LocoCtrl.cp_keys)
        self.cp = {}

        self.reset(params)

    def reset(self, params=None):
        self.in_contact['r_leg'] = 0
        self.in_contact['l_leg'] = 1

        spinal_control_phase_r = {}
        spinal_control_phase_r['ph_st'] = 0
        spinal_control_phase_r['ph_st_csw'] = 0
        spinal_control_phase_r['ph_st_sw0'] = 0
        spinal_control_phase_r['ph_st_st'] = 0
        spinal_control_phase_r['ph_sw'] = 1
        spinal_control_phase_r['ph_sw_flex_k'] = 1
        spinal_control_phase_r['ph_sw_hold_k'] = 0
        spinal_control_phase_r['ph_sw_stop_l'] = 0
        spinal_control_phase_r['ph_sw_hold_l'] = 0
        self.spinal_control_phase['r_leg'] = spinal_control_phase_r

        spinal_control_phase_l = {}
        spinal_control_phase_l['ph_st'] = 1
        spinal_control_phase_l['ph_st_csw'] = 0
        spinal_control_phase_l['ph_st_sw0'] = 0
        spinal_control_phase_l['ph_st_st'] = 0
        spinal_control_phase_l['ph_sw'] = 0
        spinal_control_phase_l['ph_sw_flex_k'] = 0
        spinal_control_phase_l['ph_sw_hold_k'] = 0
        spinal_control_phase_l['ph_sw_stop_l'] = 0
        spinal_control_phase_l['ph_sw_hold_l'] = 0
        self.spinal_control_phase['l_leg'] = spinal_control_phase_l

        self.stim['r_leg'] = dict(zip(self.m_keys, 0.01*np.ones(len(self.m_keys))))
        self.stim['l_leg'] = dict(zip(self.m_keys, 0.01*np.ones(len(self.m_keys))))

        if params is not None:
            self.set_control_params(params)

    def set_control_params(self, params):
        if len(params) == self.n_par:
            self.set_control_params_RL('r_leg', params)
            self.set_control_params_RL('l_leg', params)
        else:
            raise Exception('error in the number of params!!')

    def set_control_params_RL(self, s_leg, params):
        
        cp = {}
        cp_map = self.cp_map

        cp['theta_tgt'] = params[cp_map['theta_tgt']] *10*np.pi/180
        cp['c0'] = params[cp_map['c0']] *20*np.pi/180  +55*np.pi/180
        cp['cv'] = params[cp_map['cv']] *2*np.pi/180
        cp['alpha_delta'] = params[cp_map['alpha_delta']] *5*np.pi/180
        cp['knee_sw_tgt'] = params[cp_map['knee_sw_tgt']] *20*np.pi/180 +120*np.pi/180
        cp['knee_tgt'] = params[cp_map['knee_tgt']] *15*np.pi/180 +160*np.pi/180
        cp['knee_off_st'] = params[cp_map['knee_off_st']] *10*np.pi/180 +165*np.pi/180
        cp['ankle_tgt'] = params[cp_map['ankle_tgt']] *20*np.pi/180 +60*np.pi/180

        cp['HFL_3_PG'] = params[cp_map['HFL_3_PG']] *2.0
        cp['HFL_3_DG'] = params[cp_map['HFL_3_DG']] *1.0
        cp['HFL_6_PG'] = params[cp_map['HFL_6_PG']] *1.0
        cp['HFL_6_DG'] = params[cp_map['HFL_6_DG']] *.1
        cp['HFL_10_PG'] = params[cp_map['HFL_10_PG']] *1.0

        cp['GLU_3_PG'] = params[cp_map['GLU_3_PG']] *2.0
        cp['GLU_3_DG'] = params[cp_map['GLU_3_DG']] *0.5
        cp['GLU_6_PG'] = params[cp_map['GLU_6_PG']] *1.0
        cp['GLU_6_DG'] = params[cp_map['GLU_6_DG']] *0.1
        cp['GLU_10_PG'] = params[cp_map['GLU_10_PG']] *.5

        cp['HAM_3_GLU'] = params[cp_map['HAM_3_GLU']] *1.0
        cp['HAM_9_PG'] = params[cp_map['HAM_9_PG']] *2.0

        cp['RF_1_FG'] = params[cp_map['RF_1_FG']] *0.3
        cp['RF_8_DG_knee'] = params[cp_map['RF_8_DG_knee']] *0.1

        cp['VAS_1_FG'] = params[cp_map['VAS_1_FG']] *1.0
        cp['VAS_2_PG'] = params[cp_map['VAS_2_PG']] *2.0
        cp['VAS_10_PG'] = params[cp_map['VAS_10_PG']] *.3

        cp['BFSH_2_PG'] = params[cp_map['BFSH_2_PG']] *2.0
        cp['BFSH_7_DG_alpha'] = params[cp_map['BFSH_7_DG_alpha']] *0.2
        cp['BFSH_7_PG'] = params[cp_map['BFSH_7_PG']] *2.0
        cp['BFSH_8_DG'] = params[cp_map['BFSH_8_DG']] *1.0
        cp['BFSH_8_PG'] = params[cp_map['BFSH_8_DG']] *1.0
        cp['BFSH_9_G_HAM'] = params[cp_map['BFSH_9_G_HAM']] *2.0
        cp['BFSH_9_HAM0'] = params[cp_map['BFSH_9_HAM0']] *0.3
        cp['BFSH_10_PG'] = params[cp_map['BFSH_10_PG']] *2.0

        cp['GAS_2_FG'] = params[cp_map['GAS_2_FG']] *1.2

        cp['SOL_1_FG'] = params[cp_map['SOL_1_FG']] *1.2

        cp['TA_5_PG'] = params[cp_map['TA_5_PG']] *2.0
        cp['TA_5_G_SOL'] = params[cp_map['TA_5_G_SOL']] *0.5

        if self.control_dimension == 3:
            if len(params) != 46:
                raise Exception('error in the number of params!!')
            cp['theta_tgt_f'] = params[cp_map['theta_tgt_f']] *5.0*np.pi/180
            cp['c0_f'] = params[cp_map['c0_f']] *20*np.pi/180 + 60*np.pi/180
            cp['cv_f'] = params[cp_map['cv_f']] *10*np.pi/180
            cp['HAB_3_PG'] = params[cp_map['HAB_3_PG']] *10.0
            cp['HAB_3_DG'] = params[cp_map['HAB_3_DG']] *1
            cp['HAB_6_PG'] = params[cp_map['HAB_6_PG']] *2.0
            cp['HAD_3_PG'] = params[cp_map['HAD_3_PG']] *2.0
            cp['HAD_3_DG'] = params[cp_map['HAD_3_DG']] *0.3
            cp['HAD_6_PG'] = params[cp_map['HAD_6_PG']] *2.0
        elif self.control_dimension == 2:
            if len(params) != 37:
                raise Exception('error in the number of params!!')

        self.cp[s_leg] = cp

    def update(self, sensor_data):
        self.sensor_data = sensor_data

        if self.brain_control_on:
            self._brain_control(sensor_data)
        
        self._spinal_control(sensor_data)
        stim = np.array([self.stim['r_leg']['HFL'], self.stim['r_leg']['GLU'],
            self.stim['r_leg']['HAM'], self.stim['r_leg']['RF'],
            self.stim['r_leg']['VAS'], self.stim['r_leg']['BFSH'],
            self.stim['r_leg']['GAS'], self.stim['r_leg']['SOL'],
            self.stim['r_leg']['TA'],
            self.stim['l_leg']['HFL'], self.stim['l_leg']['GLU'],
            self.stim['l_leg']['HAM'], self.stim['l_leg']['RF'],
            self.stim['l_leg']['VAS'], self.stim['l_leg']['BFSH'],
            self.stim['l_leg']['GAS'], self.stim['l_leg']['SOL'],
            self.stim['l_leg']['TA']
            ])
        return stim

    def _brain_control(self, sensor_data=0):
        s_b = sensor_data['body']
        cp = self.cp

        self.brain_command['r_leg'] = {}
        self.brain_command['l_leg'] = {}
        for s_leg in ['r_leg', 'l_leg']:
            if self.control_dimension == 3:
                self.brain_command[s_leg]['theta_tgt_f'] = cp[s_leg]['theta_tgt_f']
                sign_frontral = 1 if s_leg is 'r_leg' else -1
                alpha_tgt_global_frontal = cp[s_leg]['c0_f'] + sign_frontral*cp[s_leg]['cv_f']*s_b['d_pos'][1]
                theta_f = sign_frontral*s_b['theta'][0]
                self.brain_command[s_leg]['alpha_tgt_f'] = alpha_tgt_global_frontal - theta_f

            self.brain_command[s_leg]['theta_tgt'] = cp[s_leg]['theta_tgt']

            alpha_tgt_global = cp[s_leg]['c0'] - cp[s_leg]['cv']*s_b['d_pos'][0]
            self.brain_command[s_leg]['alpha_tgt'] = alpha_tgt_global - s_b['theta'][1]
            self.brain_command[s_leg]['alpha_delta'] = cp[s_leg]['alpha_delta']
            self.brain_command[s_leg]['knee_sw_tgt'] = cp[s_leg]['knee_sw_tgt']
            self.brain_command[s_leg]['knee_tgt'] = cp[s_leg]['knee_tgt']
            self.brain_command[s_leg]['knee_off_st'] = cp[s_leg]['knee_off_st']
            self.brain_command[s_leg]['ankle_tgt'] = cp[s_leg]['ankle_tgt']
            self.brain_command[s_leg]['hip_tgt'] = \
                self.brain_command[s_leg]['alpha_tgt'] + 0.5*self.brain_command[s_leg]['knee_tgt']

        self.brain_command['r_leg']['swing_init'] = 0
        self.brain_command['l_leg']['swing_init'] = 0
        if sensor_data['r_leg']['contact_ipsi'] and sensor_data['l_leg']['contact_ipsi']:
            r_delta_alpha = sensor_data['r_leg']['alpha'] - self.brain_command['r_leg']['alpha_tgt']
            l_delta_alpha = sensor_data['l_leg']['alpha'] - self.brain_command['l_leg']['alpha_tgt']
            if r_delta_alpha > l_delta_alpha:
                self.brain_command['r_leg']['swing_init'] = 1
            else:
                self.brain_command['l_leg']['swing_init'] = 1
    
    def _spinal_control(self, sensor_data):
        for s_leg in ['r_leg', 'l_leg']:
            self._update_spinal_control_phase(s_leg, sensor_data)
            self.stim[s_leg] = self.spinal_control_leg(s_leg, sensor_data)

    def _update_spinal_control_phase(self, s_leg, sensor_data):
        s_l = sensor_data[s_leg]

        alpha_tgt = self.brain_command[s_leg]['alpha_tgt']
        alpha_delta = self.brain_command[s_leg]['alpha_delta']
        knee_sw_tgt = self.brain_command[s_leg]['knee_sw_tgt']

        if not self.in_contact[s_leg] and s_l['contact_ipsi']:
            self.spinal_control_phase[s_leg]['ph_st'] = 1
            self.spinal_control_phase[s_leg]['ph_sw'] = 0
            self.spinal_control_phase[s_leg]['ph_sw_flex_k'] = 0
            self.spinal_control_phase[s_leg]['ph_sw_hold_k'] = 0
            self.spinal_control_phase[s_leg]['ph_sw_stop_l'] = 0
            self.spinal_control_phase[s_leg]['ph_sw_hold_l'] = 0

        if self.spinal_control_phase[s_leg]['ph_st']:
            self.spinal_control_phase[s_leg]['ph_st_csw'] = not s_l['contact_contra']
            self.spinal_control_phase[s_leg]['ph_st_sw0'] = self.brain_command[s_leg]['swing_init']
            self.spinal_control_phase[s_leg]['ph_st_st'] = not self.spinal_control_phase[s_leg]['ph_st_sw0']

        if self.in_contact[s_leg] and not s_l['contact_ipsi']:
            self.spinal_control_phase[s_leg]['ph_st'] = 0
            self.spinal_control_phase[s_leg]['ph_st_csw'] = 0
            self.spinal_control_phase[s_leg]['ph_st_sw0'] = 0
            self.spinal_control_phase[s_leg]['ph_st_st'] = 0
            self.spinal_control_phase[s_leg]['ph_sw'] = 1
            self.spinal_control_phase[s_leg]['ph_sw_flex_k'] = 1

        if self.spinal_control_phase[s_leg]['ph_sw']:
            if self.spinal_control_phase[s_leg]['ph_sw_flex_k']:
                if s_l['phi_knee'] < knee_sw_tgt:
                    self.spinal_control_phase[s_leg]['ph_sw_flex_k'] = 0
                    self.spinal_control_phase[s_leg]['ph_sw_hold_k'] = 1
            else:
                if self.spinal_control_phase[s_leg]['ph_sw_hold_k']:
                    if s_l['alpha'] < alpha_tgt:
                        self.spinal_control_phase[s_leg]['ph_sw_hold_k'] = 0
                if s_l['alpha'] < alpha_tgt + alpha_delta:
                    self.spinal_control_phase[s_leg]['ph_sw_stop_l'] = 1
                if self.spinal_control_phase[s_leg]['ph_sw_stop_l'] \
                    and s_l['dalpha'] > 0:
                    self.spinal_control_phase[s_leg]['ph_sw_hold_l'] = 1

        self.in_contact[s_leg] = s_l['contact_ipsi']

    def spinal_control_leg(self, s_leg, sensor_data):
        s_l = sensor_data[s_leg]
        s_b = sensor_data['body']
        cp = self.cp[s_leg]

        ph_st = self.spinal_control_phase[s_leg]['ph_st']
        ph_st_csw = self.spinal_control_phase[s_leg]['ph_st_csw']
        ph_st_sw0 = self.spinal_control_phase[s_leg]['ph_st_sw0']
        ph_st_st = self.spinal_control_phase[s_leg]['ph_st_st']
        ph_sw = self.spinal_control_phase[s_leg]['ph_sw']
        ph_sw_flex_k = self.spinal_control_phase[s_leg]['ph_sw_flex_k']
        ph_sw_hold_k = self.spinal_control_phase[s_leg]['ph_sw_hold_k']
        ph_sw_stop_l = self.spinal_control_phase[s_leg]['ph_sw_stop_l']
        ph_sw_hold_l = self.spinal_control_phase[s_leg]['ph_sw_hold_l']

        theta = s_b['theta'][1]
        dtheta = s_b['dtheta'][1]

        sign_frontral = 1 if s_leg is 'r_leg' else -1
        theta_f = sign_frontral*s_b['theta'][0]
        dtheta_f = sign_frontral*s_b['dtheta'][0]

        theta_tgt = self.brain_command[s_leg]['theta_tgt']
        alpha_tgt = self.brain_command[s_leg]['alpha_tgt']
        alpha_delta = self.brain_command[s_leg]['alpha_delta']
        hip_tgt = self.brain_command[s_leg]['hip_tgt']
        knee_tgt = self.brain_command[s_leg]['knee_tgt']
        knee_sw_tgt = self.brain_command[s_leg]['knee_sw_tgt']
        knee_off_st = self.brain_command[s_leg]['knee_off_st']
        ankle_tgt = self.brain_command[s_leg]['ankle_tgt']

        stim = {}
        pre_stim = 0.01

        if self.control_dimension == 3:
            theta_tgt_f = self.brain_command[s_leg]['theta_tgt_f']
            alpha_tgt_f = self.brain_command[s_leg]['alpha_tgt_f']

            S_HAB_3 = ph_st*s_l['load_ipsi']*np.maximum(
                - cp['HAB_3_PG']*(theta_f-theta_tgt_f)
                - cp['HAB_3_DG']*dtheta_f
                , 0)
            S_HAB_6 = (ph_st_sw0*s_l['load_contra'] + ph_sw)*np.maximum(
                cp['HAB_6_PG']*(s_l['alpha_f'] - alpha_tgt_f)
                , 0)
            stim['HAB'] = S_HAB_3 + S_HAB_6

            S_HAD_3 = ph_st*s_l['load_ipsi']*np.maximum(
                cp['HAD_3_PG']*(theta_f-theta_tgt_f)
                + cp['HAD_3_DG']*dtheta_f
                , 0)
            S_HAD_6 = (ph_st_sw0*s_l['load_contra'] + ph_sw)*np.maximum(
                - cp['HAD_6_PG']*(s_l['alpha_f'] - alpha_tgt_f)
                , 0)
            stim['HAD'] = S_HAD_3 + S_HAD_6

        S_HFL_3 = ph_st*s_l['load_ipsi']*np.maximum(
            - cp['HFL_3_PG']*(theta-theta_tgt)
            - cp['HFL_3_DG']*dtheta
            , 0)
        S_HFL_6 = (ph_st_sw0*s_l['load_contra'] + ph_sw)*np.maximum(
            cp['HFL_6_PG']*(s_l['alpha']-alpha_tgt)
            + cp['HFL_6_DG']*s_l['dalpha']
            , 0)
        S_HFL_10 = ph_sw_hold_l*np.maximum(
            cp['HFL_10_PG']*(s_l['phi_hip'] - hip_tgt)
            , 0)
        stim['HFL'] = pre_stim + S_HFL_3 + S_HFL_6 + S_HFL_10

        S_GLU_3 = ph_st*s_l['load_ipsi']*np.maximum(
            cp['GLU_3_PG']*(theta-theta_tgt)
            + cp['GLU_3_DG']*dtheta
            , 0)
        S_GLU_6 = (ph_st_sw0*s_l['load_contra'] + ph_sw)*np.maximum(
            - cp['GLU_6_PG']*(s_l['alpha']-alpha_tgt)
            - cp['GLU_6_DG']*s_l['dalpha']
            , 0)
        S_GLU_10 = ph_sw_hold_l*np.maximum(
            - cp['GLU_10_PG']*(s_l['phi_hip'] - hip_tgt)
            , 0)
        stim['GLU'] = pre_stim + S_GLU_3 + S_GLU_6 + S_GLU_10

        S_HAM_3 = cp['HAM_3_GLU']*S_GLU_3
        S_HAM_9 = ph_sw_stop_l*np.maximum(
            - cp['HAM_9_PG']*(s_l['alpha'] - (alpha_tgt + alpha_delta))
            , 0)
        stim['HAM'] = pre_stim + S_HAM_3 + S_HAM_9

        S_RF_1 = (ph_st_st + ph_st_sw0*(1-s_l['load_contra']))*np.maximum(
            cp['RF_1_FG']*s_l['F_RF']
            , 0)
        S_RF_8 = ph_sw_hold_k*np.maximum(
            - cp['RF_8_DG_knee']*s_l['dphi_knee']
            , 0)
        stim['RF'] = pre_stim + S_RF_1 + S_RF_8

        S_VAS_1 = (ph_st_st + ph_st_sw0*(1-s_l['load_contra']))*np.maximum(
            cp['VAS_1_FG']*s_l['F_VAS']
            , 0)
        S_VAS_2 = -(ph_st_st + ph_st_sw0*(1-s_l['load_contra']))*np.maximum(
            cp['VAS_2_PG']*(s_l['phi_knee'] - knee_off_st)
            , 0)
        S_VAS_10 = ph_sw_hold_l*np.maximum(
            - cp['VAS_10_PG']*(s_l['phi_knee'] - knee_tgt)
            , 0)
        stim['VAS'] = pre_stim + S_VAS_1 + S_VAS_2 + S_VAS_10

        S_BFSH_2 = (ph_st_st + ph_st_sw0*(1-s_l['load_contra']))*np.maximum(
            cp['BFSH_2_PG']*(s_l['phi_knee'] - knee_off_st)
            , 0)
        S_BFSH_7 = (ph_st_sw0*(s_l['load_contra']) + ph_sw_flex_k)*np.maximum(
            - cp['BFSH_7_DG_alpha']*s_l['dalpha']
            + cp['BFSH_7_PG']*(s_l['phi_knee'] - knee_sw_tgt)
            , 0)
        S_BFSH_8 = ph_sw_hold_k*np.maximum(
            cp['BFSH_8_DG']*(s_l['dphi_knee'])
            *cp['BFSH_8_PG']*(s_l['alpha'] - alpha_tgt)
            , 0)
        S_BFSH_9 = np.maximum(
            cp['BFSH_9_G_HAM']*(S_HAM_9 - cp['BFSH_9_HAM0'])
            , 0)
        S_BFSH_10 = ph_sw_hold_l*np.maximum(
            cp['BFSH_10_PG']*(s_l['phi_knee'] - knee_tgt)
            , 0)
        stim['BFSH'] = pre_stim + S_BFSH_2 + S_BFSH_7 + S_BFSH_8 + S_BFSH_9 + S_BFSH_10

        S_GAS_2 = ph_st*np.maximum(
            cp['GAS_2_FG']*s_l['F_GAS']
            , 0)
        stim['GAS'] = pre_stim + S_GAS_2
        S_SOL_1 = ph_st*np.maximum(
            cp['SOL_1_FG']*s_l['F_SOL']
            , 0)
        stim['SOL'] = pre_stim + S_SOL_1

        S_TA_5 = np.maximum(
            cp['TA_5_PG']*(s_l['phi_ankle'] - ankle_tgt)
            , 0)
        S_TA_5_st = -ph_st*np.maximum(
            cp['TA_5_G_SOL']*S_SOL_1
            , 0)
        stim['TA'] = pre_stim + S_TA_5 + S_TA_5_st

        for muscle in stim:
            stim[muscle] = np.clip(stim[muscle], 0.01, 1.0)

        return stim

<>:269: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:388: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:269: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:388: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
C:\Users\rohan\AppData\Local\Temp\ipykernel_55580\3780282649.py:269: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  sign_frontral = 1 if s_leg is 'r_leg' else -1 # Right was 1 intially
C:\Users\rohan\AppData\Local\Temp\ipykernel_55580\3780282649.py:388: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  sign_frontral = 1 if s_leg is 'r_leg' else -1


In [ ]:
from ctrl_optim.optim.optim_utils.fati_config_parser import load_params_and_create_testenv, print_config_summary
PARAMS_FILE_PATH = r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\ctrl_optim\results\optim_results\hmedi_1500\myorfl_Kine_2D_1_25_2025Sep13_0013_None_BestLast.txt"
results_dir = os.path.dirname(PARAMS_FILE_PATH)
filename = os.path.basename(PARAMS_FILE_PATH)
bat_file_path = r"C:\Users\rohan\Downloads\Exoskeleton_PD\exo_leg\myoassist\ctrl_optim\optim\training_configs\exo_hmedi.bat"

env_wrapper, config, other = load_params_and_create_testenv(results_dir=results_dir, filename=filename,
                                                            bat_file_path=bat_file_path, sim_time=5)
base_env = getattr(env_wrapper, "env", None)

obs, info = env_wrapper.reset()

Using fatigue environment
Wrong number of params, Defaulting to 89
    MyoSuite: A contact-rich simulation suite for musculoskeletal motor control
        Vittorio Caggiano, Huawei Wang, Guillaume Durandau, Massimo Sartori, Vikash Kumar
        L4DC-2019 | https://sites.google.com/view/myosuite
    
env wrapper type: <class 'ctrl_optim.ctrl.reflex.fati_reflex_interface.myoLeg_reflex'>
base_env type: <class 'gymnasium.wrappers.time_limit.TimeLimit'>
base_env.observation_space: Box(-10.0, 10.0, (134,), float32)
base_env.observation_space.shape: (134,)


ValueError: `x` must be strictly increasing sequence.